# SAR-Assisted Understanding — analysis walkthrough

**Does radar recover what cloud destroys in optical imagery — and what is the right
way to use it?**

Three models, identical frozen encoders, identical head topology. They differ only in
what the classifier is fed:

| | classifier input | dim |
|---|---|---|
| **B** | cloud-damaged optical only | 384 |
| **C** | optical **concatenated** with raw radar | 2432 |
| **ED** | optical + radar **reconstructed into optical feature space** | 768 |

Run the pipeline first (`bash scripts/run_experiment.sh`), then execute this top to
bottom. Every number below is read from `results/metrics/`.

In [ ]:
import sys, json
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
from src.config import load_config
from src.visualization import use_style
use_style()
M = ROOT / 'results' / 'metrics'
cfg = load_config(ROOT / 'configs' / 'config.yaml')
print('levels:', cfg.masking.levels, '| seeds:', cfg.train.seeds, '| threshold:', cfg.eval.threshold)


## 1. The data

BigEarthNet v2.0 (reBEN), Lithuania summer subset. Multi-label; we keep the 11 classes
with ≥1000 positive patches.

In [ ]:
summary = json.load(open(M / 'subset_summary.json'))
manifest = pd.read_csv(M / 'subset_manifest.csv')
print(f"{summary['n_patches']} patches, {len(summary['classes'])} classes")
print('splits:', summary['split_sizes'])
print('audit problems:', summary['n_audit_problems'], 'of', summary['n_audited'], 'pairs checked')
pd.Series(summary['class_positives']).sort_values(ascending=False).to_frame('positives')


### Splits are spatially blocked, not random

Test is a contiguous interior region ringed by validation, ringed by train — so
neighbouring (highly correlated) patches almost never straddle train and test.

This looked alarming at first glance: a tile-vs-split crosstab shows both tiles in all
three splits. Plotting it spatially is what resolved it.

In [ ]:
sym = {'train': 0, 'validation': 1, 'test': 2}
fig, axes = plt.subplots(1, manifest.tile.nunique(), figsize=(11, 4.2))
for ax, (tile, g) in zip(np.atleast_1d(axes), manifest.groupby('tile')):
    grid = np.full((g.row.max()+1, g.col.max()+1), np.nan)
    grid[g.row, g.col] = g.split.map(sym)
    ax.imshow(grid, cmap='viridis', interpolation='nearest')
    ax.set_title(f'{tile}  (n={len(g)})'); ax.grid(False)

    idx = {(r, c): s for r, c, s in zip(g.row, g.col, g.split)}
    cross = tot = 0
    for (r, c), s in idx.items():
        for dr, dc in ((0, 1), (1, 0)):
            n = idx.get((r+dr, c+dc))
            if n is not None:
                tot += 1; cross += n != s
    print(f'{tile}: {cross}/{tot} adjacent pairs cross a split boundary ({cross/tot:.1%})')
plt.suptitle('Split assignment on the patch grid (dark=train, mid=val, light=test)')
plt.tight_layout(); plt.show()


## 2. The degradation is spatially coherent and exact

Masks are a deterministic function of `(patch_id, level, seed)`, so **all three arms see
byte-identical degraded imagery** — the property the whole comparison rests on.

In [ ]:
from src.masking import generate_mask
levels = [float(v) for v in cfg.masking.levels]
fig, axes = plt.subplots(1, len(levels), figsize=(15, 2.7))
for ax, lv in zip(axes, levels):
    m = generate_mask('S2B_MSIL2A_20170808T094029_N9999_R036_T35ULA_20_20', lv,
                      sigma=cfg.masking.smoothing_sigma, mask_seed=cfg.masking.mask_seed,
                      mode=cfg.masking.mode)
    ax.imshow(m, cmap='gray_r'); ax.set_title(f'{lv:.0%}  (actual {m.mean():.1%})', fontsize=9)
    ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
plt.suptitle('Simulated cloud masks — coverage is exact by construction'); plt.tight_layout(); plt.show()


## 3. Headline: three models, two regimes

**R1** trains on clean optical and evaluates degraded (the assignment's minimum
baseline). **R2** trains *every* arm with the same masking augmentation, so the only
difference between arms is what they are fed.

In [ ]:
u = pd.read_csv(M / 'unified_results.csv')
ARMS = ['optical', 'fusion', 'ed']
NAMES = {'optical': 'B optical', 'fusion': 'C fusion', 'ed': 'ED reconstruction'}

for regime, label in [('clean', 'R1 — trained on clean optical'),
                      ('degraded', 'R2 — trained with masking augmentation')]:
    piv = (u[(u.regime == regime) & (u.arm.isin(ARMS))]
           .pivot_table(index='test_level', columns='arm', values='macro_f1')[ARMS]
           .rename(columns=NAMES).round(4))
    piv.index = [f'{int(i*100)}%' for i in piv.index]
    print(f'\n{label}'); print(piv.to_string())

**Read the two regimes against each other.** In R1 radar looks transformative
(+0.27 at 80% masking). In R2, where both arms get the same augmentation, the same radar
branch buys **+0.037**. Roughly **90% of the apparent benefit of radar was the benefit of
training on degraded data at all.**

Note also that **ED is worse than C under R1** — a head trained only on clean features
never learns to lean on the reconstruction. The two ideas are not independent.

In [ ]:
from IPython.display import Image, display
display(Image(filename=str(ROOT / 'results/figures/07_three_model_curves.png')))

## 4. Which way of using radar wins?

A first 3-seed run put ED above C at 80% by +0.0074; an identical rerun gave −0.0060.
MPS kernels are non-deterministic and the gap is smaller than the run-to-run spread, so
this was redone with **10 seeds, paired arm-to-arm** (`scripts/09_ed_compare.py`).

In [ ]:
m = pd.read_csv(M / 'ed_matched_comparison.csv')
m[['masking_pct', 'B_mean', 'C_mean', 'ED_mean', 'ED_minus_C', 'ED_minus_C_std',
   'ED_beats_C_n_seeds', 'seed_ci_excludes_zero']].round(4)

Three distinct regimes, and the honest summary needs all three:

1. **0–60% masking — reconstruction wins**, +0.017 to +0.024, 10/10 seeds agreeing.
2. **80% — a dead heat.** +0.0012 on 5/10 seeds. No claim to make here.
3. **100% — reconstruction fails badly**, −0.305, 0/10 seeds.

**Why ED wins low:** arm C hands the head 2432 numbers, 2048 of them radar, and when the
optical image is barely damaged those add nothing — at 0% masking C is actually *worse*
than plain optical. ED compresses radar to 384 dimensions already aligned with the
optical space.

**Why ED fails at 100%:** `z_degraded` is a constant there, so half the ED input is dead,
and the head was trained only on levels ≤0.8. A design flaw shared with C, not a property
of reconstruction.

In [ ]:
display(Image(filename=str(ROOT / 'results/figures/08_three_model_deltas.png')))

### The capacity control

`fusion_shuf` is arm C with radar features permuted across samples: identical parameter
count, no genuine pairing. It scores *below* optical-only everywhere, so arm C's
behaviour is not explained by extra head capacity alone.

In [ ]:
d = u[u.regime == 'degraded']
ctrl = (d[d.arm.isin(['optical', 'fusion', 'ed', 'fusion_shuf', 'sar'])]
        .pivot_table(index='test_level', columns='arm', values='macro_f1').round(4))
ctrl.index = [f'{int(i*100)}%' for i in ctrl.index]
ctrl

## 5. Per-class: the two radar routes keep different things

In [ ]:
pc = pd.read_csv(M / 'unified_per_class.csv')
t = pc[(pc.regime == 'degraded') & (pc.masking_pct == 80) & (pc.arm.isin(ARMS))]
piv = t.pivot_table(index='class', columns='arm', values='f1')[ARMS]
piv.columns = ['B', 'C', 'ED']
piv['C - B'] = (piv.C - piv.B).round(3)
piv['ED - B'] = (piv.ED - piv.B).round(3)
piv['ED - C'] = (piv.ED - piv.C).round(3)
piv.round(3).sort_values('ED - C', ascending=False)

The classes where **raw concatenation still beats reconstruction** are exactly those
with distinctive radar signatures that have **no optical analogue**:

* **Urban fabric** (ED − C = −0.057) — buildings meeting the ground form a corner
  reflector, giving a bright *double-bounce* return. A geometry fact, not a colour fact.
* **Inland waters** (−0.017) — calm water is a mirror at radar wavelengths, so almost
  nothing returns. Again geometry, not colour.

Forcing radar through a "predict the optical feature" bottleneck necessarily discards
these. That is the conceptual cost of the reconstruction framing, and it shows up in the
numbers rather than needing to be argued.

*(Marine waters is the largest ED gain and the least trustworthy number here: all 116
Marine-waters test patches are exactly the 116 test patches from the second tile.)*

In [ ]:
display(Image(filename=str(ROOT / 'results/figures/09_three_model_per_class.png')))
display(Image(filename=str(ROOT / 'results/figures/10_three_model_error_matrices.png')))

## 6. Does radar actually predict the clean optical feature?

This is the question ED is built to answer, and it needs its own metric. The obvious one
is a trap.

In [ ]:
zc_tr = np.load(ROOT / 'data/features/train_opt_L000.npy')
zc_te = np.load(ROOT / 'data/features/test_opt_L000.npy')
cos = lambda a, b: ((a*b).sum(1) / (np.linalg.norm(a,axis=1)*np.linalg.norm(b,axis=1)+1e-12))
rec = pd.read_csv(M / 'ed_reconstruction.csv')

mu = zc_tr.mean(0, keepdims=True)
perm = np.random.default_rng(0).permutation(len(zc_te))
print(f"cos(two unrelated patches)   = {cos(zc_te, zc_te[perm]).mean():.4f}   <- anisotropy floor")
print(f"cos(constant mean predictor) = {cos(np.repeat(mu, len(zc_te), 0), zc_te).mean():.4f}")
print(f"cos(the decoder)             = {rec[rec.variant=='ed'].cos_hat_vs_clean.mean():.4f}")
print(f"\nfraction of feature energy in the mean vector = "
      f"{np.linalg.norm(mu)**2 / (zc_te**2).sum(1).mean():.3f}")

Raw cosine of 0.987 looks like a solved problem. It is not: **93% of the feature
energy is in the dataset mean**, two unrelated patches already sit at 0.958, and a
constant predictor scores 0.976. The decoder's entire achievement is the gap from 0.976
to 0.987.

So the honest metrics are **mean-relative**: `R² = 1 − MSE/MSE_mean` and centred cosine.
And the decisive test is the control the study already uses for arm C — break the
correspondence and see what survives.

In [ ]:
g  = rec[rec.variant == 'ed'].groupby('test_level').mean(numeric_only=True)
gs = rec[rec.variant == 'ed_shuf'].groupby('test_level').mean(numeric_only=True)
pd.DataFrame({
    'R2': [g.r2_hat.mean(), gs.r2_hat.mean()],
    'centred cos': [g.centered_cos_hat.mean(), gs.centered_cos_hat.mean()],
    'raw cos': [g.cos_hat_vs_clean.mean(), gs.cos_hat_vs_clean.mean()],
}, index=['decoder on real radar', 'decoder on SHUFFLED radar']).round(4)

The shuffled control collapses **exactly** onto the constant-mean predictor
(R² ≈ 0). So the real decoder's R² = 0.457 is genuine, patch-specific learning: radar
predicts ~46% of the patch-to-patch variance in the clean optical feature.

Note what raw cosine did — it moved 0.011 between a model that learned *nothing* and one
that learned a real mapping. It could not have told them apart.

In [ ]:
r = g[['r2_degraded', 'r2_hat']].copy()
r.index = [f'{int(i*100)}%' for i in r.index]
r['z_hat closer to clean?'] = np.where(r.r2_hat > r.r2_degraded, 'yes', 'no')
r.round(3)

`ẑ_clean` is flat because radar is never masked. `z_degraded` decays and goes sharply
negative — past ~40% masking the ViT's own output is **further from the clean feature than
guessing the dataset average**. Crossover at ≈15% masking.

This is feature-space distance, not classification accuracy — and §4 showed classification
does not simply follow, which is exactly why both are measured.

In [ ]:
display(Image(filename=str(ROOT / 'results/figures/06_encoder_decoder.png')))

## 7. Failure analysis: the aggregate tie hides large disagreement

ED and C differ by +0.0012 at 80% masking, which reads as "the same model". Look at the
patch level instead.

In [ ]:
from src.visualization import sample_f1
z  = np.load(M / 'test_probs.npz', allow_pickle=True)
ze = np.load(M / 'ed_test_probs.npz', allow_pickle=True)
thr, t = float(cfg.eval.threshold), z['targets']
fb = sample_f1(t, (z['degraded/optical/0.8'] >= thr).astype(int))
fc = sample_f1(t, (z['degraded/fusion/0.8']  >= thr).astype(int))
fe = sample_f1(t, (ze['ed/ed_r2/0.8']        >= thr).astype(int))

print(f'ED strictly better than C : {(fe - fc > 0.05).sum():4d}  ({(fe-fc > 0.05).mean():.1%})')
print(f'C strictly better than ED : {(fc - fe > 0.05).sum():4d}  ({(fc-fe > 0.05).mean():.1%})')
print(f'tied within 0.05          : {(np.abs(fe-fc) <= 0.05).sum():4d}  ({(np.abs(fe-fc) <= 0.05).mean():.1%})')

fig, axes = plt.subplots(1, 2, figsize=(12, 3.6))
axes[0].hist(fc - fb, bins=60, color='#eb6834'); axes[0].set_title('C - B  (raw radar vs none)')
axes[1].hist(fe - fc, bins=60, color='#8256d0'); axes[1].set_title('ED - C  (reconstruction vs concatenation)')
for ax in axes:
    ax.axvline(0, color='k', lw=1); ax.set_xlabel('per-patch F1 difference at 80% masking')
axes[0].set_ylabel('patches'); plt.tight_layout(); plt.show()

The two radar routes disagree on **more than half of all patches**, in almost
perfectly balanced directions. The aggregate tie is two large opposing effects
cancelling, not agreement.

That is a far more interesting result than "no difference", and it is the strongest
argument for **routing** between them on estimated cloud fraction rather than picking a
winner.

Also worth stating plainly: at the patch level arm C **repairs 1.7%** of patches and
**breaks 3.0%** relative to B — it damages nearly twice as many as it fixes, while still
improving macro F1, because the repairs land in rare classes that macro-averaging weights
heavily.

In [ ]:
display(Image(filename=str(ROOT / 'results/figures/11_three_model_qualitative_L080.png')))

Selection rules are fixed in the code before any result is inspected, and two of the
four categories are cases where reconstruction is the *worse* choice.

**Row 1** is the clearest case for reconstruction: a lakeside patch where radar shows a
large black region (specular water). Arm B predicts one wrong label, arm C predicts
*nothing* above threshold, and ED recovers *Inland waters* at 0.64. The radar evidence was
equally available to C — routing it through the optical bottleneck is what made it usable.

**Row 2** is the clearest case against: conifer/mixed forest, where C gets both labels and
ED gets only one. Structural canopy distinctions do not survive projection into optical
feature space.

## 8. Conclusion

1. **Optical degradation is devastating for a model that never saw it** (0.70 → 0.28 at
   80% masking) and mild for one that did (0.69 → 0.60). Masking augmentation alone
   recovers ~9× what radar contributes.
2. **Against the fair baseline, concatenated radar helps only past ~60% cloud**
   (+0.037 at 80%, bootstrap CI excludes zero) and is neutral-to-negative below it.
3. **Reconstruction is the better route below ~60% cloud** (+0.017 to +0.024, 10/10
   seeds), ties at 80%, and fails at 100%.
4. **Radar genuinely predicts the clean optical representation** — R² = 0.457 against a
   shuffled-radar control at −0.005 — but raw cosine (0.987) is a trap in this feature
   space and would have overstated it badly.
5. **Neither radar route dominates.** They disagree on 52% of patches in balanced
   directions. The useful next step is routing between them, not choosing one.

**Biggest caveats:** 2 Sentinel-2 tiles, 2 acquisition dates, one country, one season —
the effective sample size for any geographic generalisation claim is closer to two
satellite passes than to 8,774 images. Synthetic cloud masks with hard edges and no
shadow. The 100% column is out of distribution for every masking-trained head. And all
116 Marine-waters test patches are exactly the 116 test patches from the second tile, so
that class is entangled with geography.